# Dorso-ventral morphospace

One combined publication figure (`morphospace`): where species sit in the UNICOM
embedding seen from above and below, how varied genera are on each side, how tightly
the two sides covary, and how intraspecific variation relates to the dorso-ventral
difference.

Reads the tables `morphospace integrate` wrote into the backend database
(`packages/morphospace`), read-only; nothing is recomputed here. Each species is
represented by the mean UNICOM embedding of its dorsal and of its ventral photographs
(species with at least three photographs of a side; accepted names from the
harmonized taxonomy). The PCA of panel A is fitted on both sides' centroids stacked,
so the two sides share axes. Disparity (B) is the sum of variances of species
centroids in the full 768-dimensional embedding, rarefied to a common number of
species; integration (C) is the Mantel correlation between the dorsal and ventral
species distance matrices.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "backend/app/configs/config.yaml").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
from analyses.helpers.morphospace import (
    disparity_panel,
    integration_panel,
    morphospace_panel,
    morphospace_summaries,
    variation_panel,
)
from analyses.helpers.publication import (
    align_panel_titles,
    export_figure,
    load_settings,
    publication_style,
)

settings = load_settings(ROOT)
publication_style()

# The scope panel A draws: the whole collection ("all", "all"), or one family,
# e.g. ("family", "nymphalidae").
SCOPE_RANK, SCOPE_KEY = "all", "all"

In [ ]:
summaries = morphospace_summaries(settings, scope_rank=SCOPE_RANK, scope_key=SCOPE_KEY)
scope = summaries["scope"].iloc[0]
print(f"Run {scope['run_id']}: {scope['scope_name']}, {scope['n_species']:,} species "
      f"({scope['n_species_both']:,} seen from both sides)")

## Combined morphospace figure

A) Species centroids on the shared PC1 × PC2; dorsal filled, ventral hollow, segments
join a species' two sides for the coloured groups. B) One point per genus with ≥3
species: rarefied dorsal against ventral disparity with 95% intervals; points above
the dashed 1:1 line have the more varied underside. C) Genus dorso-ventral
integration against the number of species seen from both sides; the dashed line is
the integration of the whole scope. D) Per species, intraspecific dispersion (mean
cosine distance of photographs to their centroid, averaged over sides) against the
cosine distance between the dorsal and ventral centroids.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11), layout="constrained")
(ax_a, ax_b), (ax_c, ax_d) = axes

morphospace_panel(ax_a, summaries["points"], scope)
ax_a.set_title(
    f"A) Dorso-ventral morphospace\nN = {scope['n_species']:,} species", loc="left"
)
disparity_panel(ax_b, summaries["disparity"])
ax_b.set_title(
    f"B) Surface disparity by genus\nN = {len(summaries['disparity']):,} genera", loc="left"
)
integration_panel(ax_c, summaries["integration"], scope["dv_mantel_r"])
ax_c.set_title(
    f"C) Dorso-ventral integration\nN = {len(summaries['integration']):,} genera", loc="left"
)
variation_panel(ax_d, summaries["species"])
ax_d.set_title(
    f"D) Intraspecific variation and dorso-ventral divergence\n"
    f"N = {len(summaries['species']):,} species",
    loc="left",
)

fig.canvas.draw()
fig.set_layout_engine(None)
align_panel_titles(axes.ravel())

export_figure(
    fig,
    settings,
    "morphospace",
    {
        "points": summaries["points"],
        "disparity": summaries["disparity"],
        "integration": summaries["integration"],
        "species": summaries["species"],
    },
)
plt.show()
plt.close(fig)